## Загрузка и использование модели Deep-Seek OCR
Установление окружения ollama, загрузка модели и тестовый прогон изображений с последующей валидацией для формата obsidian markdown

In [ ]:
!pip install langchain-ollama
!pip install ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama)
thread.start()



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (561 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently i

In [ ]:
import ollama

model_name = "deepseek-ocr:3b"
print("Pulling...")
ollama.pull(model_name)
print("Pulled")

image_path = "/content/images/IMG_20260504_133519.jpg"

response = ollama.chat(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": "<|grounding|>Convert the document to markdown.",
            "images": [image_path],
        }
    ],
)
print("Answer:")
print(response)

Pulling...


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
import ollama

model_name = "deepseek-ocr:3b"
print("Pulling...")
ollama.pull(model_name)
print("Pulled")

Pulling...
Pulled


In [ ]:
response.message.content

"\\> Теорема 4 (о дифференцировании по параметру). Пусть функция \\( f(x, y) \\) непрерывна в прямоугольнике \\( K=\\{(x, y): a \\le x \\le b, c \\le y \\le d\\} \\) и имеет непрерывную частную производную \\( \\frac{\\partial f(x, y)}{\\partial y} \\) в области \\( G \\) такой, что \\( K \\subset G \\).<br/>\n\\> Тогда \\( F(y) \\) есть непрерывно дифференцируемая функция параметра \\( y \\) на отрезке \\([c, d]\\), причем\n\\[ \\frac{\\partial}{\\partial y} \\int_a^b f(x, y) dx = \\int_a^b \\frac{\\partial f(x, y)}{\\partial y} dy. \\]\n\\> В общем случае, когда пределы интегрирования есть дифференцируемые функции \\( \\phi(y) \\) и \\( \\psi(y) \\) параметра \\( y \\) и \\( a < \\phi(y) < b, a < \\psi(y) < b \\), имеет место формула:\n\\[ \\frac{\\partial}{\\partial y} \\int_a^b f(x, y) dx = \\int_a^b \\frac{\\partial f(x, y)}{\\partial y} \\frac{\\partial \\phi(y)}{\\partial y} dy + f(\\psi(y), y) \\psi'(y) - f(\\phi(y), y) \\phi'(y). \\]\n\\> Девятиркова М.В., 2026\n\\> Актуальная

In [ ]:
import re


def clean_deepseek_ocr(text: str) -> str:
    text = re.sub(r"<\|(?:im_start|im_end|md_start|md_end)\|>", "", text)
    text = re.sub(
        r"</?\|?(?:assistant|user|system)\|?>", "", text, flags=re.IGNORECASE
    )

    text = re.sub(r"\\>\s*", "> ", text)

    text = re.sub(
        r"\\\[(.*?)\\\]",
        lambda m: "$$\n" + m.group(1).strip() + "\n$$",
        text,
        flags=re.DOTALL,
    )

    text = re.sub(
        r"\\\((.*?)\\\)",
        lambda m: "$" + m.group(1).strip() + "$",
        text,
        flags=re.DOTALL,
    )

    text = re.sub(r"\\([{}])", r"\1", text)

    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [ ]:
clean_deepseek_ocr(response.message.content)

NameError: name 'response' is not defined

## Разметка данных
Тестовая разметка одного батча при помощи Deep-Seek OCR

In [ ]:
from pathlib import Path
def deep_seek_response(file: Path):
  response = ollama.chat(
      model=model_name,
      messages=[
          {
              "role": "user",
              "content": "<|grounding|>Convert the document to markdown.",
              "images": [file],
          }
      ],
  )
  raw_text = response.message.content
  text = clean_deepseek_ocr(raw_text)
  return text

In [ ]:
def get_tokens(text, alphabet):
  math_block_regex = re.compile(r'\$\$(.*?)\$\$|\$(.*?)\$', re.DOTALL)
  latex_command_regex = re.compile(r'\\[a-zA-Z]+')
  math_blocks = math_block_regex.findall(text)

  for block in math_blocks:
    block_content = block[0] if block[0] else block[1]
    if block_content:
      commands = latex_command_regex.findall(block_content)
      alphabet.update(commands)

In [ ]:
import string

eng_letters = string.ascii_letters
digits = string.digits
rus_letters = "абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
alphabet= set(eng_letters + rus_letters + digits)

In [ ]:
import shutil
import zipfile
from pathlib import Path
import pandas as pd

images_path = Path("/content/drive/MyDrive/Мат. Анализ/Мат. Анализ")
extracted_path = Path("/content/extracted_archives")
extracted_path.mkdir(parents=True, exist_ok=True)
csv_path = Path("/content/drive/MyDrive/data.csv")

archive_extensions = {".zip", ".tar", ".gz", ".rar", ".7z"}
for file in images_path.rglob("*"):
  if file.is_file() and file.suffix.lower() in archive_extensions:
    target_dir = extracted_path / file.stem
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
      if file.suffix.lower() == ".zip":
        with zipfile.ZipFile(file, "r") as zip_ref:
          zip_ref.extractall(target_dir)
      else:
        shutil.unpack_archive(str(file), str(target_dir))
    except Exception as e:
      print(f"Не удалось распаковать {file}: {e}")


processed_paths = set()
if csv_path.exists():
  try:
    existing_df = pd.read_csv(csv_path)
    if "img_path" in existing_df.columns:
      processed_paths = set(existing_df["img_path"].astype(str))
  except Exception:
    pass

# 2. Рекурсивный сбор картинок с защитой от повторной обработки
image_extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff"}
search_dirs = [images_path, extracted_path]

for base_dir in search_dirs:
  if not base_dir.exists():
    continue
  for file in base_dir.rglob("*"):
    if file.is_file() and file.suffix.lower() in image_extensions:
      file_str = str(file)

      if file_str in processed_paths:
        continue

      text = deep_seek_response(file)
      tokens = get_tokens(text, alphabet)

      row_df = pd.DataFrame([{"img_path": file_str, "text": text}])
      row_df.to_csv(
          csv_path,
          mode="a",
          header=not csv_path.exists() or csv_path.stat().st_size == 0,
          index=False,
      )

      processed_paths.add(file_str)

## Тест на время
Проверка времени за которое модель Deep-Seek OCR справляется с батчом